# Multi-Factor Investment Classification with TOPSIS and VIKOR

**Coding for Finance** — Università degli Studi di Bergamo

A junior quant analyst is asked to classify a universe of stocks on return, risk and
risk-adjusted performance, then rank them with two independent multi-criteria decision
methods — **TOPSIS** and **VIKOR** — to see whether they agree on the best pick.

**Data.** The exercise sheet's original 50-stock dataset was not available, so this
notebook uses a real, sourced 20-stock universe instead (tickers, 1-year total return,
ChartRow, Sep 2026). The method is unchanged.

## 1. Classification

Each stock is first labelled with a simple rule-based system combining Return, Risk and the Sharpe Ratio, plus a Mean-Variance score:

$$\text{MV Score} = \mu - \lambda\sigma^2 \qquad (\lambda = 3 \text{, risk-aversion coefficient})$$

| Class | Rule |
|---|---|
| Excellent | Return > 10%, Risk < 0.18, Sharpe > 1.5 |
| Good | (Return ≥ 5% and Risk < 0.18) or (Sharpe > 1.0 and Return ≥ 5%) |
| Risky | Return ≥ 5% and Risk ≥ 0.18 |
| Moderate | Return 2–5% and Risk 0.18–0.22 |
| Avoid | otherwise |

This part uses an explicit loop over stocks (the classification rules are most naturally expressed row-by-row); the TOPSIS/VIKOR ranking further down is fully vectorized.

In [1]:
import numpy as np
import pandas as pd

stocks = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA",
          "META", "NVDA", "JPM", "V", "NFLX",
          "ORCL", "INTC", "AMD", "GS", "BAC",
          "WMT", "KO", "PEP", "DIS", "CSCO"]

returns = np.array([
    0.3394, -0.0082, 0.4621, 0.0969, 0.0455,
    -0.1735, 0.3437, 0.2030, 0.0881, -0.3778,
    -0.2799, 2.8931, 1.9518, 0.4133, 0.2535,
    0.0998, 0.3323, -0.0155, -0.1010, 0.6389
])

# "Risk" here is a simplified cross-sectional dispersion measure (each
# stock's squared deviation from the mean return across the 20 stocks),
# not a time-series volatility -- consistent with the exercise's own data.
risk_free_rate = 0.0479
variances = (returns - returns.mean()) ** 2
risks = np.sqrt(variances)
sharpe_ratios = (returns - risk_free_rate) / risks

In [2]:
lambda_risk_aversion = 3
mv_scores = returns - lambda_risk_aversion * variances
mv_scores_normalized = (mv_scores - mv_scores.min()) / (mv_scores.max() - mv_scores.min())

return_high, risk_low, sharpe_good, sharpe_excellent = 0.05, 0.18, 1.0, 1.5

investment_class, mv_efficient = [], []
for i in range(len(stocks)):
    r, risk, sharpe = returns[i], risks[i], sharpe_ratios[i]
    if r > 0.10 and risk < risk_low and sharpe > sharpe_excellent:
        investment_class.append("Excellent Investment"); mv_efficient.append("Yes (High)")
    elif (r >= return_high and risk < risk_low) or (sharpe > sharpe_good and r >= return_high):
        investment_class.append("Good Investment"); mv_efficient.append("Yes")
    elif r >= return_high and risk >= risk_low:
        investment_class.append("Risky Investment"); mv_efficient.append("No")
    elif 0.02 <= r < return_high and risk_low <= risk <= 0.22:
        investment_class.append("Moderate"); mv_efficient.append("Maybe")
    else:
        investment_class.append("Avoid"); mv_efficient.append("No")

summary_df = pd.DataFrame({
    "Stock": stocks, "Return": returns, "Risk": risks, "Sharpe Ratio": sharpe_ratios,
    "MV Score": mv_scores, "MV Normalized": mv_scores_normalized,
    "Classification": investment_class, "MV Efficient": mv_efficient
})
print(summary_df.round(4))

    Stock  Return    Risk  Sharpe Ratio  MV Score  MV Normalized  \
0    AAPL  0.3394  0.0209       13.9641    0.3381         0.9945   
1    MSFT -0.0082  0.3685       -0.1522   -0.4155         0.9496   
2   GOOGL  0.4621  0.1018        4.0678    0.4310         1.0000   
3    AMZN  0.0969  0.2634        0.1860   -0.1112         0.9677   
4    TSLA  0.0455  0.3148       -0.0076   -0.2517         0.9593   
5    META -0.1735  0.5338       -0.4148   -1.0282         0.9131   
6    NVDA  0.3437  0.0166       17.8462    0.3429         0.9947   
7     JPM  0.2030  0.1573        0.9862    0.1288         0.9820   
8       V  0.0881  0.2722        0.1477   -0.1341         0.9663   
9    NFLX -0.3778  0.7381       -0.5768   -2.0121         0.8544   
10   ORCL -0.2799  0.6402       -0.5120   -1.5094         0.8844   
11   INTC  2.8931  2.5328        1.1233  -16.3525         0.0000   
12    AMD  1.9518  1.5915        1.1963   -5.6471         0.6379   
13     GS  0.4133  0.0530        6.8911    0.404

## 2. TOPSIS ranking

**TOPSIS** ranks alternatives by geometric distance to an ideal point across *all* criteria
at once: Return, Risk (cost), Sharpe Ratio and MV Score, equally weighted (0.25 each).

$$\text{Closeness}_i = \frac{D_i^-}{D_i^+ + D_i^-}$$

where $D_i^+$ and $D_i^-$ are each stock's Euclidean distance to the ideal-best and ideal-worst vectors. Higher closeness = better.

In [3]:
criteria = np.column_stack([returns, risks, sharpe_ratios, mv_scores])
is_benefit = [True, False, True, True]   # Risk is the only cost criterion
weights = np.array([0.25, 0.25, 0.25, 0.25])

norm_matrix = criteria / np.sqrt((criteria ** 2).sum(axis=0))       # vector normalization
weighted = norm_matrix * weights

ideal_best = np.where(is_benefit, weighted.max(axis=0), weighted.min(axis=0))
ideal_worst = np.where(is_benefit, weighted.min(axis=0), weighted.max(axis=0))

dist_best = np.sqrt(((weighted - ideal_best) ** 2).sum(axis=1))
dist_worst = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))

topsis_score = dist_worst / (dist_best + dist_worst)
topsis_rank = np.argsort(-topsis_score)

print("TOPSIS ranking (best to worst)")
for rank, i in enumerate(topsis_rank, start=1):
    print(rank, stocks[i], "Closeness Coefficient:", round(topsis_score[i], 4))

TOPSIS ranking (best to worst)
1 NVDA Closeness Coefficient: 0.6726
2 AAPL Closeness Coefficient: 0.6562
3 KO Closeness Coefficient: 0.6326
4 GS Closeness Coefficient: 0.6152
5 GOOGL Closeness Coefficient: 0.5954
6 CSCO Closeness Coefficient: 0.5847
7 BAC Closeness Coefficient: 0.5635
8 JPM Closeness Coefficient: 0.5511
9 WMT Closeness Coefficient: 0.5333
10 AMZN Closeness Coefficient: 0.5329
11 V Closeness Coefficient: 0.5315
12 TSLA Closeness Coefficient: 0.5251
13 MSFT Closeness Coefficient: 0.5171
14 PEP Closeness Coefficient: 0.516
15 AMD Closeness Coefficient: 0.5067
16 DIS Closeness Coefficient: 0.5032
17 META Closeness Coefficient: 0.492
18 ORCL Closeness Coefficient: 0.4749
19 NFLX Closeness Coefficient: 0.4583
20 INTC Closeness Coefficient: 0.3921


## 3. VIKOR ranking

**VIKOR** instead balances *group utility* $S$ (how well a stock does on average across
criteria) against *individual regret* $R$ (its single worst-performing criterion), combined via $v = 0.5$:

$$Q_i = v\cdot\frac{S_i - S^*}{S^- - S^*} + (1-v)\cdot\frac{R_i - R^*}{R^- - R^*}$$

Lower $Q$ = better compromise solution.

In [4]:
v = 0.5
f_star = np.where(is_benefit, criteria.max(axis=0), criteria.min(axis=0))
f_minus = np.where(is_benefit, criteria.min(axis=0), criteria.max(axis=0))

denom = f_star - f_minus
denom[denom == 0] = 1e-12
norm_dist = weights * (f_star - criteria) / denom

S = norm_dist.sum(axis=1)
R = norm_dist.max(axis=1)
S_star, S_minus = S.min(), S.max()
R_star, R_minus = R.min(), R.max()

Q = (v * (S - S_star) / (S_minus - S_star) +
     (1 - v) * (R - R_star) / (R_minus - R_star))
vikor_rank = np.argsort(Q)

print("VIKOR ranking (best to worst, by Q -- lower is better)")
for rank, i in enumerate(vikor_rank, start=1):
    print(rank, stocks[i], "S:", round(S[i], 4), "R:", round(R[i], 4), "Q:", round(Q[i], 4))

print()
print("TOPSIS top pick:", stocks[topsis_rank[0]], " |  VIKOR top pick:", stocks[vikor_rank[0]])

VIKOR ranking (best to worst, by Q -- lower is better)
1 NVDA S: 0.1962 R: 0.1949 Q: 0.0625
2 AAPL S: 0.2497 R: 0.1952 Q: 0.1155
3 GS S: 0.3422 R: 0.1895 Q: 0.1579
4 KO S: 0.3026 R: 0.1957 Q: 0.1697
5 GOOGL S: 0.3812 R: 0.187 Q: 0.1744
6 CSCO S: 0.4121 R: 0.2134 Q: 0.413
7 BAC S: 0.4299 R: 0.216 Q: 0.4508
8 JPM S: 0.4529 R: 0.2288 Q: 0.5736
9 AMD S: 0.5449 R: 0.2259 Q: 0.6376
10 WMT S: 0.4852 R: 0.2395 Q: 0.6887
11 AMZN S: 0.486 R: 0.2396 Q: 0.6909
12 V S: 0.4884 R: 0.2402 Q: 0.6973
13 TSLA S: 0.4997 R: 0.2423 Q: 0.7247
14 MSFT S: 0.5136 R: 0.2442 Q: 0.7533
15 PEP S: 0.5154 R: 0.2445 Q: 0.7568
16 DIS S: 0.537 R: 0.2466 Q: 0.7937
17 META S: 0.5553 R: 0.2478 Q: 0.8209
18 ORCL S: 0.5825 R: 0.2491 Q: 0.857
19 NFLX S: 0.6081 R: 0.25 Q: 0.888
20 INTC S: 0.7269 R: 0.25 Q: 1.0

TOPSIS top pick: NVDA  |  VIKOR top pick: NVDA


## 4. TOPSIS vs. VIKOR: do they agree?

TOPSIS ranks by geometric distance to an ideal point across all criteria at once, while
VIKOR explicitly balances "majority of criteria" group utility ($S$) against the single
worst-performing criterion for each stock ($R$) via the parameter $v$. When the two
methods agree on the top pick, that stock is a robust choice; when they disagree, it
usually means the stock does very well on most criteria (favoring TOPSIS's
majority-style ranking) but has one weak criterion that VIKOR's $R$ term penalizes more
heavily. In practice, cross-checking a ranking with a second method like this is a cheap
way to flag picks that only look good under one particular way of aggregating criteria.